<a href="https://colab.research.google.com/github/programminghistorian/ph-submissions/blob/gh-pages/assets/texto-a-vectores-word-embeddings/texto-a-vectores-word-embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica: explorando novelas hispanoamericanas del siglo XIX

### Importación de librerías

In [4]:
# Carga de las librerías que se utilizarán durante la práctica
import re                     # para expresiones regulares
import os                     # para consultar información sobre el sistema operativo
import string                 # para manipular cadenas de caracteres
import glob                   # para localizar un tipo de archivo específico
from pathlib import Path      # para acceder a archivos en otros directorios
import gensim                 # para acceder a Word2Vec
from gensim.models import Word2Vec    # para acceder a la versión de Word2Vec de Gensim
import pandas as pd           # para ordenar y organizar datos

Es posible que tengas que instalar algunas de las bibliotecas que se utilizan en esta lección. Ejemplo: !pip install gensim

### Obtención de datos

In [62]:
#dirpath = r'FILL IN YOUR FILE PATH HERE'      # COMPLETAR CON LA RUTA A LA CARPETA txt/
dirpath = r'/content/txt/'   # COMPLETAR CON LA RUTA A LA CARPETA txt/
file_type = ".txt"           # si tus datos no están en formato de texto plano, puedes cambiar esto
filenames = []
data = []

# recorrer todos los archivos del directorio indicado
for file in glob.glob(os.path.join(dirpath, '*' + file_type)):
    with open(file, 'r', encoding='utf-8') as f:
        text = f.read()
        data.append(text)
        filenames.append(Path(file).stem)

print(f"Se encontraron {len(filenames)} archivos.")
print("Primeros 5 nombres de archivo:", filenames[:5])
print("Primeros 500 caracteres del primer archivo:", data[0][:500])

Se encontraron 38 archivos.
Primeros 5 nombres de archivo: ['nh0027', 'nh0031', 'nh0041', 'nh0040', 'nh0054']
Primeros 500 caracteres del primer archivo: Dos años hacía que mi tío vivía en mi compañía cuando de pronto una mañana al sentarnos a almorzar, me dijo:
            
—Sobrino: me caso...
            
Cualquiera creería que me dio la noticia con acento enérgico. ¡Muy lejos de eso! Su voz fue como siempre suave, e insinuante como un arrullo, pues mi tío, aunque tenía el carácter del zorro, afectaba siempre la mansedumbre del cordero.
            
¿Y qué tenía de particular que mi tío se casara? ¡Vaya si lo tenía! Había cumplido los cincuent


Nota: A diferencia del corpus original en inglés, los archivos del conha19 están codificados en UTF-8 e incluyen caracteres propios del español (tildes, ñ, signos de apertura de interrogación y exclamación, etc.). Por eso es importante especificar encoding='utf-8' al abrir cada archivo, como se muestra en el bloque anterior.


### Limpieza del corpus

In [36]:
def clean_text(text):
    # Separa el texto en tokens y lo convierte a minúsculas
    tokens = text.split()
    tokens = [t.lower() for t in tokens]

    # Elimina puntuación, incluyendo signos de apertura propios del español (¿ y ¡)
    # que no forman parte de string.punctuation de Python 3 de manera predeterminada
    extra_punct = '¿¡'
    re_punc = re.compile('[%s]' % re.escape(string.punctuation + extra_punct))
    tokens = [re_punc.sub('', token) for token in tokens]

    # elimina números y tokens vacíos; isalpha() reconoce ñ y vocales acentuadas en UTF-8
    tokens = [token for token in tokens if token.isalpha()]
    return tokens

data_clean = []
for x in data:
    data_clean.append(clean_text(x))

# comprueba que el número de textos procesados sea el mismo que el de los textos originales
print(len(data))
print(len(data_clean))

# comprueba que el primer token del primer texto procesado coincida con el original
print(data[0].split()[0])
print(data_clean[0][0])

# comprueba que el último token del primer texto procesado coincida con el original
print(data[0].split()[-1])
print(data_clean[0][-1])

76
76
Muchas
muchas
amor...
amor


### Creación del modelo

In [34]:
# Entrena el modelo
model = Word2Vec(sentences=data_clean, window=5, min_count=3, workers=4, epochs=5, sg=1)
# Guarda el modelo
model.save("word2vec_conha19.model")


In [39]:
# Entrenar el modelo con otros parámetros
#model2 = Word2Vec(sentences=data_clean, window=4, min_count=6, workers=4, epochs=5, sg=1)
# Guarda el modelo
#model.save("word2vec_conha19_2.model")


### Interrogando al modelo mediante consultas exploratorias


In [40]:
# Comprueba si una palabra existe en nuestro vocabulario
word = "honor"

if word in model.wv.key_to_index:
    print("La palabra %s está en el vocabulario del modelo" % word)
else:
    print("%s no está en el vocabulario del modelo" % word)


La palabra honor está en el vocabulario del modelo


In [41]:
# Devuelve las diez palabras usadas en contextos más similares a "patria"
model.wv.most_similar('patria', topn=10)


[('escuchó', 0.7854388356208801),
 ('recomendándoles', 0.7613583207130432),
 ('herida', 0.7551472187042236),
 ('ligaba', 0.7484256029129028),
 ('alejamiento', 0.7209076285362244),
 ('material', 0.7002909779548645),
 ('lazo', 0.6960281729698181),
 ('par', 0.6930586695671082),
 ('comunicaran', 0.6910613775253296),
 ('cualquier', 0.6732879877090454)]

In [42]:
# Devuelve las diez palabras usadas en contextos similares a "amor" que no comparten el contexto de "matrimonio"
model.wv.most_similar(positive=["amor"], negative=["matrimonio"], topn=10)


[('descuajaba', 0.35962164402008057),
 ('opima', 0.35276153683662415),
 ('remitiendo', 0.3524825870990753),
 ('propio', 0.33935609459877014),
 ('cuestión', 0.33065345883369446),
 ('prorrumpiendo', 0.32441845536231995),
 ('cedo', 0.3163539469242096),
 ('prima', 0.3130801320075989),
 ('irlandesa', 0.31182363629341125),
 ('italianas', 0.3057747483253479)]

In [43]:
# Para explorar cómo se habla de mujer en relación con sociedad en el corpus:
# Devuelve las diez palabras usadas en contextos más similares a la combinación de "mujer" y "sociedad"

model.wv.most_similar(positive=["mujer", "sociedad"], topn=10)



[('ganaba', 0.7212947010993958),
 ('mister', 0.7067298889160156),
 ('apoyo', 0.7062670588493347),
 ('pariente', 0.6963048577308655),
 ('aspiraba', 0.6923717260360718),
 ('cantalicia', 0.6908117532730103),
 ('osa', 0.6893097162246704),
 ('distinguida', 0.6881428956985474),
 ('acudir', 0.6880282163619995),
 ('reunidos', 0.6861887574195862)]

In [44]:
# Devuelve una puntuación de similitud coseno para las dos palabras que introduzcas. en este caso, "honor" y "vergüenza"
model.wv.similarity("honor", "vergüenza")


np.float32(0.6313442)

In [45]:
# Devuelve una predicción para las demás palabras de una frase que contengan las palabras "amor", "pasión" y "virtud"
model.predict_output_word(["amor", "pasión", "virtud"])


[('amor', np.float32(0.0009077215)),
 ('corazón', np.float32(0.00036449675)),
 ('prima', np.float32(0.0003529767)),
 ('esa', np.float32(0.00028491937)),
 ('inés', np.float32(0.0002841965)),
 ('propio', np.float32(0.00028188076)),
 ('atrevido', np.float32(0.00027834295)),
 ('vida', np.float32(0.00027033107)),
 ('mi', np.float32(0.00026195138)),
 ('palabra', np.float32(0.00026108357))]

In [ ]:
# Define una lista de pares de palabras para probar la similitud semántica.
# Se usará para evaluar cómo el modelo Word2Vec relaciona estas palabras.

In [46]:
test_words = [('amor', 'pasión'),
              ('patria', 'nación'),
              ('honor', 'virtud'),
              ('muerte', 'sepulcro'),
              ('indio', 'mestizo'),
              ('ciudad', 'pueblo')]

### Validación del modelo


In [60]:

# COMPLETAR CON LA RUTA A LA CARPETA DONDE GUARDASTE LOS MODELOS
# Ejemplo en Google Colab: r'/content/'
# Ejemplo en Windows:      r'C:/Users/tuusuario/Documents/conha19/'
# Ejemplo en Mac/Linux:    r'/Users/tuusuario/Documents/conha19/'

models_folder = r'/content/'

# buscar todos los archivos .model en la carpeta indicada
model_files = list(Path(models_folder).glob('*.model'))
print(f"Se encontraron {len(model_files)} modelos: {[f.name for f in model_files]}")

model_list = []
model_filenames = []

for filename in model_files:
    file_path = str(filename)
    print(f"Cargando modelo: {file_path}")
    model = Word2Vec.load(file_path)
    model_list.append(model)
    model_filenames.append(file_path)

# pares de palabras de prueba para evaluar los modelos
# seleccionados a partir del vocabulario de la narrativa hispanoamericana del siglo XIX
test_words = [('amor', 'pasión'),
              ('patria', 'nación'),
              ('honor', 'virtud'),
              ('muerte', 'sepulcro'),
              ('indio', 'mestizo'),
              ('ciudad', 'pueblo')]

# crear un dataframe vacío con los encabezados de columna necesarios
evaluation_results = pd.DataFrame(columns=['Modelo', 'Palabras de prueba', 'Similitud coseno'],
                                   dtype=object)

# contador acumulativo para el índice del dataframe
row_index = 0

# iterar por model_list
for i in range(len(model_list)):
    # para cada modelo, evaluar todos los pares de palabras
    for x in range(len(test_words)):
        # verificar que ambas palabras del par estén en el vocabulario del modelo
        word1, word2 = test_words[x]
        if word1 in model_list[i].wv and word2 in model_list[i].wv:
            # calcular la puntuación de similitud coseno para el par
            similarity_score = model_list[i].wv.similarity(word1, word2)
        else:
            # si alguna palabra no está en el vocabulario, registrar None
            similarity_score = None
            print(f"Advertencia: '{word1}' o '{word2}' no están en el vocabulario de {model_filenames[i]}")

        # agregar la fila al dataframe usando el índice acumulativo
        evaluation_results.loc[row_index] = [model_filenames[i], test_words[x], similarity_score]
        row_index += 1

# guardar evaluation_results como archivo .csv
evaluation_results.to_csv('evaluacion_modelo_word2vec.csv', index=False)


Se encontraron 2 modelos: ['word2vec_conha19.model', 'word2vec_conha19_2.model']
Cargando modelo: /content/word2vec_conha19.model
Cargando modelo: /content/word2vec_conha19_2.model


In [61]:
# Imprimir los resultados de la evaluación
print("\nResultados de la evaluación:")
print(evaluation_results)


Resultados de la evaluación:
                               Modelo  Palabras de prueba  Similitud coseno
0     /content/word2vec_conha19.model      (amor, pasión)          0.666444
1     /content/word2vec_conha19.model    (patria, nación)          0.528450
2     /content/word2vec_conha19.model     (honor, virtud)          0.816222
3     /content/word2vec_conha19.model  (muerte, sepulcro)          0.596927
4     /content/word2vec_conha19.model    (indio, mestizo)          0.775680
5     /content/word2vec_conha19.model    (ciudad, pueblo)          0.272473
6   /content/word2vec_conha19_2.model      (amor, pasión)          0.666444
7   /content/word2vec_conha19_2.model    (patria, nación)          0.528450
8   /content/word2vec_conha19_2.model     (honor, virtud)          0.816222
9   /content/word2vec_conha19_2.model  (muerte, sepulcro)          0.596927
10  /content/word2vec_conha19_2.model    (indio, mestizo)          0.775680
11  /content/word2vec_conha19_2.model    (ciudad, pueblo) 